In Loan_approval6.ipynb I am working from Loan_approval4.pynb and changing the house "ownership" to label encoding.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import tensorflow as tf



In [ ]:
# Read the JSON file into a Pandas DataFrame
df = pd.read_json("../data/Loan_approval_dataset.json")

df.head()

In [ ]:
df.describe()


In [ ]:
df.dtypes

In [ ]:
unique_states_count = len(df["STATE"].unique())
print(unique_states_count)

In [ ]:
state_app_counts = df["STATE"].value_counts()
state_app_counts

In [ ]:
# place some states (<2000 applications) into an "OTHER" column to reduce dimensionality
states_to_replace = state_app_counts[state_app_counts < 2000].index.tolist()

for state in states_to_replace:
    df["STATE"] = df["STATE"].replace(state, "Other")

df["STATE"].value_counts()

In [ ]:
unique_cities_count = len(df["CITY"].unique())
print(unique_cities_count)

In [ ]:
# Drop the cities column - see if using just the states as geographic area works: to reduce dimensionality
df = df.drop("CITY", axis=1)

In [ ]:
df["Profession"].value_counts()

In [ ]:
# make a copy named df_original to retain the original dataframe
df_original = df

In [ ]:
df["Profession"]=df["Profession"].replace({"Industrial_Engineer":"Engineer","Mechanical_engineer":"Engineer","Chemical_engineer":"Engineer","Biomedical_Engineer":"Engineer",
                                         "Petroleum_Engineer":"Engineer","Civil_Engineer":"Engineer"})
#df["Profesion"]=df["Profession"].replace({"Industrial_Engineer ":"Engineer","Mechanical_engineer":"Engineer","Chemical_engineer":"Engineer","Biomedical_Engineer":"Engineer",
   #                                      "Petroleum_Engineer":"Engineer","Civil_Engineer":"Engineer"})
df["Profession"].value_counts()


In [ ]:
df.head()

In [ ]:
# reindex the data frame to set the Id as the index
df.set_index("Id", inplace=True)

In [ ]:
# loop through values in Married/Single column to make a new column with 0 for married, 1 for single
df = pd.get_dummies(df, columns=["Married/Single"],dtype= int)

df.head()

In [ ]:
# Encode Ownership columns with get_dummies

df= pd.get_dummies(df, columns=["Car_Ownership"],dtype= int)
df.head()

In [ ]:
# Use label encoder to encode the house ownership column
label_encoder = LabelEncoder()
df['House_Ownership'] = label_encoder.fit_transform(df['House_Ownership'])

In [ ]:
# drop the married column  and the Car_Ownership no column to reduce the dimensionality, but retain binary nature of this information
df = df.drop("Married/Single_married", axis=1)
df = df.drop("Car_Ownership_no", axis=1)
df.head()

In [ ]:
# Encode the profession column with get_dummies
df= pd.get_dummies(df, columns=["Profession"],dtype= int)

In [ ]:
# Encode the State column with git_dummies
df=pd.get_dummies(df, columns=["STATE"], dtype=int)
df.head()

In [ ]:
# Separate the data into labels and features
# Separate the y variable, the labels
y = df["Risk_Flag"]

# Separate the X variable, the features
X = df.drop("Risk_Flag", axis=1)

In [ ]:
df.head()

In [ ]:
# Split the data using train_test_split
# Assign a random_state of 5 to the function
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=5, stratify=y)

In [ ]:
# Create a MinMaxScaler instances
scaler = MinMaxScaler()

# Fit the MinMaxScaler
X_scaler = scaler.fit(X_train)

# Scale the data
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

In [ ]:
# Define the model - deep neural net, i.e., the number of input features and hidden nodes for each layer.
inputs = 74
layer1_units = 150
layer2_units = 90
layer3_units = 30
layer4_units = 16
layer1_activation = "relu"
layer2_activation = "relu"
layer3_activation = "tanh"
layer4_activation = "tanh"

nn = tf.keras.models.Sequential()

# First hidden layer
nn.add(tf.keras.layers.Dense(units = layer1_units, activation = layer1_activation, input_dim = inputs))

# Second hidden layer
nn.add(tf.keras.layers.Dense(units = layer2_units, activation = layer2_activation))

# Third hidden layer
nn.add(tf.keras.layers.Dense(units = layer3_units, activation = layer2_activation))

#Fourth hidden layer
nn.add(tf.keras.layers.Dense(units = layer4_units, activation = layer4_activation))

# Output layer
nn.add(tf.keras.layers.Dense(units = 1, activation = "sigmoid"))

# Check the structure of the model
nn.summary()

In [ ]:
# Compile the model
nn.compile(loss = "binary_crossentropy", optimizer = "adam", metrics=["accuracy"])

import math
batch_size=32
num_batches= len(X_train_scaled)/batch_size
num_batches= math.ceil(num_batches)
num_batches

In [ ]:
# Train the model
nn_fit = nn.fit(X_train_scaled, y_train, epochs=50)

In [ ]:
model_loss, model_accuracy = nn.evaluate(X_test_scaled, y_test, verbose=2)
print(f"loss: {model_loss}, accuracy: {model_accuracy}")

In [ ]:
import pickle
pickle.dump(nn, open('model.pkl','wb'))

In [ ]:
import joblib
joblib.dump(nn, '../data/model_joblib.pkl')
joblib.dump(MinMaxScaler, '../data/scaler.pkl')

In [ ]:
# To load the model
# model = joblib.load('../data/model.pkl')
# scaler = joblib.load('../data/model.pkl')